# Lifespan Predictor — Survival Analysis on SageMaker

Predict **time to event** — when a machine will fail, when a customer
will churn, how long a patient will survive — from historical data in
which some subjects have not yet had the event.

This notebook runs end to end with no edits and no data of your own.
It generates its own sample dataset, trains a model, tunes it, creates
a SageMaker model, runs batch transforms, and deletes what it made.

It runs on either Lifespan Predictor product, including the **free
tier**: it trains on 100 units, within the free tier's cap of
100 units. See [Which product?](https://github.com/kooditurva/lifespan-predictor-sdk#which-product) for the
two products and their algorithm ARNs.

**What you need**: an active AWS Marketplace subscription to either
product, and a SageMaker execution role.

## 1. Install the SDK

The client SDK is installed from its GitHub release tag. It is not
published to PyPI.

In [ ]:
# Other releases: https://github.com/kooditurva/lifespan-predictor-sdk/releases
!pip install -q "lifespan-predictor @ git+https://github.com/kooditurva/lifespan-predictor-sdk@v1.0.2"

## 2. Point at the algorithm you subscribed to

The ARN below is the free tier's, in `eu-west-1`. If you subscribed to
the other product or work in another Region, use the ARN listed under
[Which product?](https://github.com/kooditurva/lifespan-predictor-sdk#which-product), or find it in the AWS Marketplace
console: **Manage subscriptions**, select the product, choose
**Continue to configuration**, and pick your Region.

The execution role is the IAM role SageMaker assumes to run your
training jobs. The SDK reads it from `SAGEMAKER_ROLE_ARN`. Outside
SageMaker Studio, set that variable before starting Jupyter; in Studio,
the cell below sets it from `get_execution_role()`.

In [ ]:
import os

import numpy as np
import pandas as pd

from lifespan_predictor_sdk import SurvivalModel, ContinuousRange

ALGORITHM_ARN = 'arn:aws:sagemaker:eu-west-1:985815980388:algorithm/free-tier-lifespan-predictor-m-227e5356082e3b6ea8f8e5e080c1ef12'

if 'SAGEMAKER_ROLE_ARN' not in os.environ:
    from sagemaker import get_execution_role

    os.environ['SAGEMAKER_ROLE_ARN'] = get_execution_role()

### Check your setup

What the SDK will use. Set `SAGEMAKER_BUCKET` to keep the data and models
in a bucket of your own; otherwise SageMaker's default bucket for your
account and Region is used, and created if needed.

In [ ]:
from importlib.metadata import version

import boto3

region = boto3.Session().region_name
arn_region = ALGORITHM_ARN.split(':')[3]

print('SDK version:        ', version('lifespan-predictor'))
print('Region:             ', region)
print('AWS_PROFILE:        ', os.environ.get('AWS_PROFILE', '(not set)'))
print('SAGEMAKER_ROLE_ARN: ', os.environ['SAGEMAKER_ROLE_ARN'])
print('SAGEMAKER_BUCKET:   ', os.environ.get(
    'SAGEMAKER_BUCKET', "(not set: SageMaker's default bucket)"))
print('Algorithm ARN:      ', ALGORITHM_ARN)

if region != arn_region:
    print(f'Warning: the session is in {region}, but the algorithm is in '
          f'{arn_region}. Use the ARN for your Region.')

## 3. Generate a sample dataset

So the notebook is self-contained, we synthesise a small survival
dataset rather than downloading one.

The shape the algorithm expects:

| column | meaning |
|---|---|
| `unit` | the subject — a machine, a customer, a patient |
| `cycles` | the step within a unit's history: `1, 2, 3, …` |
| `duration` | how long it was observed |
| `event` | `1` if the event happened, `0` if still censored |
| everything else | features |

**Censoring** is the point of survival analysis: a row with `event = 0`
has not failed *yet*, which is information, not a missing value.

In [ ]:
rng = np.random.default_rng(0)
n_units = 125

# Two features that genuinely drive the risk, plus noise.
load = rng.uniform(0.2, 1.0, n_units)
temperature = rng.normal(70, 12, n_units)
noise = rng.normal(0, 1, n_units)

# Higher load and temperature shorten the lifetime.
baseline = 120 - 60 * load - 0.4 * (temperature - 70)
true_lifetime = np.clip(baseline + rng.normal(0, 8, n_units), 5, None)

# Some units are still running when observation stops — censored.
censor_at = rng.uniform(30, 140, n_units)
duration = np.minimum(true_lifetime, censor_at)
event = (true_lifetime <= censor_at).astype(int)

# One observation per unit here, so every unit is at step 1.
df = pd.DataFrame({
    'unit': np.arange(n_units),
    'cycles': 1,
    'duration': duration.round(1),
    'event': event,
    'load': load.round(3),
    'temperature': temperature.round(2),
    'noise': noise.round(3),
})

# Train on 100 units, the free tier's cap; hold out the rest.
train_df = df.iloc[:100].reset_index(drop=True)
test_df = df.iloc[100:].reset_index(drop=True)

print(f'{len(train_df)} training units, {len(test_df)} held out')
print(f"{event.mean():.0%} had the event; the rest are censored")
df.head()

## 4. Train

`fit()` submits a SageMaker training job against the subscribed
algorithm. Nothing about the model runs locally.

`model_type='deepsurv'` trains DeepSurv, a neural network that
generalises the Cox proportional hazards model.

In [ ]:
model = SurvivalModel({
    'algorithm_arn': ALGORITHM_ARN,
    'model_type': 'deepsurv',
    'instance_type': 'ml.m5.large',
    'num_nodes': [32, 32],
    'epochs': 20,
})

log = model.fit(train_df)
model.print_training_job_summary()

## 5. Tune

Put a **range** where a value would go and `fit()` switches to
hyperparameter search — there is no separate tune method. `n_trials`
sets the budget.

Ranges available: `ContinuousRange(low, high, log=False)`,
`IntegerRange(low, high)` and `CategoricalRange([...])`.

The search runs inside one training job, on the same units as above, so
the free tier allows it.

In [ ]:
tuned = SurvivalModel({
    'algorithm_arn': ALGORITHM_ARN,
    'model_type': 'deepsurv',
    'instance_type': 'ml.m5.large',
    'num_nodes': [32, 32],
    'epochs': 20,
    'learning_rate': ContinuousRange(1e-4, 1e-2, log=True),
    'dropout': ContinuousRange(0.0, 0.5),
    'n_trials': 4,
})

tuned.fit(train_df, val_split=0.2)

trials = tuned.get_tuning_trials()
print('Best configuration:', trials.best_config)
for trial in trials:
    print(trial)

## 6. Create a SageMaker model

`deploy()` turns the trained estimator into a SageMaker **Model**, which
is what batch transform runs against. Section 9 hosts it on a real-time
endpoint instead.

In [ ]:
model_name = tuned.deploy()
print('Created SageMaker model:', model_name)

## 7. Batch transform

`predict_mean()` runs a batch transform job and returns the expected
time to event for each row — in the same units as `duration`.

For the full survival curve rather than a single number, use
`predict()`, which returns survival probability over time.

In [ ]:
features = test_df.drop(columns=['duration', 'event'])

predictions = tuned.predict_mean(features)

results = test_df[['unit', 'duration', 'event']].copy()
results['predicted_time_to_event'] = np.asarray(predictions).ravel().round(1)
results.head(10)

### Survival curves

`predict()` runs a second batch transform and returns, for each row,
the probability of having survived to each point in time: one column
per row, indexed by time.

In [ ]:
import matplotlib.pyplot as plt

few = features.head(5)
curves = tuned.predict(few)

ax = curves.plot(drawstyle='steps-post', figsize=(8, 4))
ax.set_xlabel('time')
ax.set_ylabel('survival probability')
ax.legend([f'unit {u}' for u in few['unit']])
plt.show()

## 8. Score it

**C-index** measures ranking: given two units, how often does the model
put the one that failed first at higher risk? `0.5` is a coin flip.

**RMSE** measures the error in predicted time, computed over units that
actually had the event — a censored unit has no true time to compare to.

In [ ]:
evaluation = tuned.evaluate(
    features,
    durations=test_df['duration'].values,
    events=test_df['event'].values,
)
print(f'C-index: {evaluation.c_index:.3f}')
print(f'RMSE:    {evaluation.rmse:.1f}')

## 9. Real-time endpoint

`deploy(endpoint_instance_type=...)` hosts the model on an endpoint,
which answers in seconds instead of starting a job. `predict()` and
`predict_mean()` then go through it, with the same results as batch
transform. An endpoint is billed for as long as it exists, so the next
section deletes it.

In [ ]:
tuned.deploy(endpoint_instance_type='ml.m5.large')

realtime = tuned.predict(few)
difference = float((realtime - curves).abs().max().max())
print(f'Largest difference from batch transform: {difference:.2e}')

## 10. Clean up

Delete the endpoint and the SageMaker model this notebook created, so
trying the product does not leave anything behind.

Training jobs are billed for the time they ran and stop on their own;
there is nothing to delete for those.

In [ ]:
tuned.delete()
print('Deleted', model_name)

## Where to go next

- **Your own data** — one row per observation of a unit, with `unit`,
  `duration`, `event` and your features. On the free tier, train on at
  most 100 units.
- **Sensor histories** — several rows per `unit` over time work too;
  `rolling_means` adds windowed averages of each feature, which helps
  when signals drift before failure.

Full documentation is in the SDK repository:
https://github.com/kooditurva/lifespan-predictor-sdk